# PRIDE — Notebook 3: Decoy Distinguishability Attack

Fills **Table VI** of the revised manuscript.

Implements the adversary A_dist from Section VII.B.3:
  - A_dist receives K+1 candidate plaintexts (one real, K decoys, unknown order)
    plus a held-out sample from P for training.
  - A_dist trains a logistic-regression + random-forest ensemble classifier.
  - Empirical attack success rate p̂_att = fraction of trials where A_dist
    correctly identifies the real plaintext.

Expected result under identical distributions (Synthetic Gaussian row):
  p̂_att ≈ 1/(K+1)  (uniform random guess; confirms Theorem 1bis)

All datasets auto-download. No manual file placement required.

## Cell 1 — Installs (run once)

In [6]:
import subprocess, sys
def _install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ('ucimlrepo', 'scikit-learn'):
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        print(f"Installing {pkg} ..."); _install(pkg)

print("Dependencies OK.")

Installing scikit-learn ...
Dependencies OK.


## Cell 2 — Imports

In [9]:
import os, sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# __file__ is undefined in Jupyter; fall back to cwd (works in both contexts)
sys.path.insert(0, str(__import__('pathlib').Path(
    globals().get('__file__', '.')).resolve().parent))

from pride_core import (
    PRIDE, PRIDEParams, GaussianSampler, EmpiricalSampler,
    load_adult, load_har, load_airquality,
)

SEED    = 40
N_TRIALS = 1_000    # trials per (dataset, K) cell — matches the paper
N_TRAIN  = 500      # held-out training samples given to the adversary
K_VALUES = [1, 5, 10]
N_THR    = 3        # threshold n

rng = np.random.default_rng(SEED)
print(f"Config: N_TRIALS={N_TRIALS}, N_TRAIN={N_TRAIN}, K_values={K_VALUES}, n={N_THR}")

Config: N_TRIALS=1000, N_TRAIN=500, K_values=[1, 5, 10], n=3


## Cell 3 — Dataset loaders

In [12]:
print("Loading UCI Adult ...")
adult_vals = load_adult()
print(f"  Adult: {len(adult_vals)} records")

print("\nLoading UCI HAR ...")
har_vals, har_labels = load_har()
print(f"  HAR PCA-1: {len(har_vals)} records")

print("\nLoading UCI Air Quality ...")
air_vals = load_airquality()
print(f"  Air Quality CO: {len(air_vals)} records")

print("\nAll datasets loaded.")

Loading UCI Adult ...
  [cached] Adult age: N=48842
  Adult: 48842 records

Loading UCI HAR ...
  [cached] HAR: N=10299
  HAR PCA-1: 10299 records

Loading UCI Air Quality ...
  [cached] Air Quality CO: N=7674, mean=2.153
  Air Quality CO: 7674 records

All datasets loaded.


## Cell 4 — Adversary A_dist

The adversary is given:
  - X_train: N_TRAIN samples drawn from P (held-out, not encrypted)
  - X_cands: K+1 plaintexts in shuffled order (one real, K decoys)

A_dist trains a one-class / binary classifier on X_train vs a few random
decoy examples, then ranks X_cands by their "real-ness" score.
Top-1 accuracy = p̂_att.

In [15]:
def run_attack_cell(name: str, real_vals: np.ndarray,
                    decoy_sampler_factory, K: int,
                    n_trials: int, n_train: int,
                    n: int, seed: int) -> dict:
    """
    Run n_trials adversarial trials for a given dataset and K.

    Returns dict with: Dataset, K, p_att, baseline (1/(K+1)).
    """
    rng_local = np.random.default_rng(seed)
    params = PRIDEParams(n=n, K=K, init_shares=n + 2)
    system = PRIDE(params, seed=seed)
    sampler = decoy_sampler_factory(seed + 42)
    ct = system.setup()

    correct = 0
    for trial in range(n_trials):
        # --- Encrypt one real record with K decoys ---
        m_real_f = float(rng_local.choice(real_vals))
        m_real_i = int(round(m_real_f)) % params.p
        sks = system.enc(ct, m_real_i, sampler)

        # --- Decrypt all K+1 plaintexts (in unknown shuffled order) ---
        plaintexts_int = [system.dec(ct, sk) for sk in sks]
        p_field = params.p
        # Bring back to original scale (undo mod wrap)
        plaintexts = [
            float(v) if v <= p_field // 2 else float(v - p_field)
            for v in plaintexts_int
        ]
        # Shuffle so real position (index 0) is hidden from adversary
        real_pos = 0                       # known to us, not to A_dist
        order = rng_local.permutation(K + 1)
        shuffled = [plaintexts[o] for o in order]
        real_shuffled_pos = int(np.where(order == 0)[0][0])

        # --- Adversary trains on held-out sample from P ---
        train_idx = rng_local.choice(len(real_vals), size=n_train, replace=False)
        X_train = real_vals[train_idx].reshape(-1, 1)
        # Adversary also generates "decoy" training examples to build a binary clf
        decoy_train = np.array(
            [float(sampler()) for _ in range(n_train)]
        ).reshape(-1, 1)
        X_fit = np.vstack([X_train, decoy_train])
        y_fit = np.concatenate([np.ones(n_train), np.zeros(n_train)])

        scaler = StandardScaler()
        X_fit_s = scaler.fit_transform(X_fit)
        X_cands_s = scaler.transform(np.array(shuffled).reshape(-1, 1))

        # Logistic regression (fast) + random forest (stronger)
        try:
            lr = LogisticRegression(max_iter=500, random_state=seed)
            lr.fit(X_fit_s, y_fit)
            lr_scores = lr.predict_proba(X_cands_s)[:, 1]

            rf = RandomForestClassifier(n_estimators=20, random_state=seed,
                                        n_jobs=1)
            rf.fit(X_fit_s, y_fit)
            rf_scores = rf.predict_proba(X_cands_s)[:, 1]

            # Ensemble: average scores
            scores = (lr_scores + rf_scores) / 2.0
        except Exception:
            # Fall back to LR only if RF fails (tiny dataset edge case)
            scores = lr_scores

        guess_pos = int(np.argmax(scores))
        if guess_pos == real_shuffled_pos:
            correct += 1

    p_att = correct / n_trials
    baseline = 1.0 / (K + 1)
    return {
        'Dataset': name,
        'K': K,
        'p_att': round(p_att, 4),
        'baseline_1_over_K1': round(baseline, 4),
        'excess': round(p_att - baseline, 4),
    }

print("Adversary function defined.")

Adversary function defined.


## Cell 5 — Decoy sampler factories

Must match the decoy generators used in the real-data notebook (realdata.py)
so that the attack and the TV proxy are measuring the same decoy quality.

In [18]:
def adult_factory(seed): return EmpiricalSampler(adult_vals, jitter=0.0, seed=seed)
def har_factory(seed):   return EmpiricalSampler(har_vals,   jitter=float(har_vals.std()) * 0.05, seed=seed)
def air_factory(seed):   return EmpiricalSampler(air_vals,   jitter=float(air_vals.std()) * 0.05, seed=seed)
def gauss_factory(seed): return GaussianSampler(mean=1000.0, std=100.0, seed=seed)

DATASETS = [
    ('Synthetic Gaussian', np.random.default_rng(SEED).normal(1000, 100, 100_000), gauss_factory),
    ('Adult',              adult_vals,                                               adult_factory),
    ('HAR',                har_vals,                                                 har_factory),
    ('Air Quality',        air_vals,                                                 air_factory),
]
print("Dataset list defined.")

Dataset list defined.


## Cell 6 — Run the attack experiment

Total: 4 datasets × 3 K-values = 12 cells.
Each cell runs N_TRIALS=1,000 adversarial trials.
Expected runtime: 5–15 minutes on a modern laptop.

In [21]:
results = []
total = len(DATASETS) * len(K_VALUES)
done = 0

for ds_name, ds_vals, ds_factory in DATASETS:
    for K in K_VALUES:
        row = run_attack_cell(
            name=ds_name, real_vals=ds_vals,
            decoy_sampler_factory=ds_factory,
            K=K, n_trials=N_TRIALS, n_train=N_TRAIN,
            n=N_THR, seed=SEED
        )
        done += 1
        print(f"[{done:2d}/{total}] {ds_name:<20} K={K:2d}  "
              f"p̂_att={row['p_att']:.4f}  "
              f"baseline={row['baseline_1_over_K1']:.4f}  "
              f"excess={row['excess']:+.4f}")
        results.append(row)

df = pd.DataFrame(results)
print("\nAll cells complete.")

[ 1/12] Synthetic Gaussian   K= 1  p̂_att=0.4960  baseline=0.5000  excess=-0.0040
[ 2/12] Synthetic Gaussian   K= 5  p̂_att=0.1690  baseline=0.1667  excess=+0.0023
[ 3/12] Synthetic Gaussian   K=10  p̂_att=0.0810  baseline=0.0909  excess=-0.0099
[ 4/12] Adult                K= 1  p̂_att=0.5000  baseline=0.5000  excess=+0.0000
[ 5/12] Adult                K= 5  p̂_att=0.1690  baseline=0.1667  excess=+0.0023
[ 6/12] Adult                K=10  p̂_att=0.1060  baseline=0.0909  excess=+0.0151
[ 7/12] HAR                  K= 1  p̂_att=0.4950  baseline=0.5000  excess=-0.0050
[ 8/12] HAR                  K= 5  p̂_att=0.1580  baseline=0.1667  excess=-0.0087
[ 9/12] HAR                  K=10  p̂_att=0.0830  baseline=0.0909  excess=-0.0079
[10/12] Air Quality          K= 1  p̂_att=0.4880  baseline=0.5000  excess=-0.0120
[11/12] Air Quality          K= 5  p̂_att=0.1700  baseline=0.1667  excess=+0.0033
[12/12] Air Quality          K=10  p̂_att=0.0930  baseline=0.0909  excess=+0.0021

All cells compl

## Cell 7 — Sanity checks

Under the identical-distribution assumption (Synthetic Gaussian row),
p̂_att must be ≈ 1/(K+1). Excess should be ≈ 0.

In [23]:
print("\nSanity check — Synthetic Gaussian (excess should be ≈ 0):")
sg = df[df['Dataset'] == 'Synthetic Gaussian']
for _, row in sg.iterrows():
    ok = abs(row['excess']) < 0.05
    flag = "OK" if ok else "CHECK"
    print(f"  K={row['K']:2d}: p̂_att={row['p_att']:.4f}, "
          f"baseline={row['baseline_1_over_K1']:.4f}, excess={row['excess']:+.4f}  [{flag}]")


Sanity check — Synthetic Gaussian (excess should be ≈ 0):
  K= 1: p̂_att=0.4960, baseline=0.5000, excess=-0.0040  [OK]
  K= 5: p̂_att=0.1690, baseline=0.1667, excess=+0.0023  [OK]
  K=10: p̂_att=0.0810, baseline=0.0909, excess=-0.0099  [OK]


## Cell 8 — Table VI output

In [25]:
print("\n" + "="*80)
print("TABLE VI: Empirical attack success rate p̂_att vs K")
print("="*80)

pivot = df.pivot_table(index='Dataset', columns='K', values='p_att')
baselines = df.pivot_table(index='Dataset', columns='K', values='baseline_1_over_K1')

header = f"{'Dataset':<22}" + "".join(f"  K={K:2d}" for K in K_VALUES)
print(header)
print("-"*80)
for ds in [d[0] for d in DATASETS]:
    row_str = f"{ds:<22}"
    for K in K_VALUES:
        val = pivot.loc[ds, K] if ds in pivot.index else float('nan')
        row_str += f"  {val:.4f}"
    print(row_str)
print("-"*80)
baseline_str = f"{'Baseline 1/(K+1)':<22}"
for K in K_VALUES:
    baseline_str += f"  {1.0/(K+1):.4f}"
print(baseline_str)
print("="*80)


TABLE VI: Empirical attack success rate p̂_att vs K
Dataset                 K= 1  K= 5  K=10
--------------------------------------------------------------------------------
Synthetic Gaussian      0.4960  0.1690  0.0810
Adult                   0.5000  0.1690  0.1060
HAR                     0.4950  0.1580  0.0830
Air Quality             0.4880  0.1700  0.0930
--------------------------------------------------------------------------------
Baseline 1/(K+1)        0.5000  0.1667  0.0909


## Cell 9 — LaTeX snippet for Table VI

In [27]:
print("\nLaTeX rows for Table VI (replace [run] placeholders):\n")
for ds_name, _, _ in DATASETS:
    cells = []
    for K in K_VALUES:
        row = df[(df['Dataset'] == ds_name) & (df['K'] == K)].iloc[0]
        cells.append(f"{row['p_att']:.4f}")
    baseline_K10 = f"{1.0/11:.3f}"
    print(f"{ds_name} & " + " & ".join(cells) + f" & {baseline_K10} \\\\ \\hline")


LaTeX rows for Table VI (replace [run] placeholders):

Synthetic Gaussian & 0.4960 & 0.1690 & 0.0810 & 0.091 \\ \hline
Adult & 0.5000 & 0.1690 & 0.1060 & 0.091 \\ \hline
HAR & 0.4950 & 0.1580 & 0.0830 & 0.091 \\ \hline
Air Quality & 0.4880 & 0.1700 & 0.0930 & 0.091 \\ \hline
